# Module 00: Interactive Environment & Tooling Walkthrough

Welcome to the **interactive notebook** for Module 00! This notebook allows you to experiment with Python's runtime introspection, understand virtual environment isolation, explore how Python searches for packages (`sys.path`), and see the mechanics of modern Python tooling.

## 1. Python Interpreter & Runtime Introspection
Let's inspect the active Python interpreter executing this kernel.

In [ ]:
import os
import platform
import sys

print("=== Active Python Interpreter ===")
print(f"Version          : {sys.version.split()[0]}")
print(f"Full Info        : {sys.version}")
print(f"Executable Path  : {sys.executable}")
print(f"Platform OS      : {platform.platform()}")
print(f"Architecture     : {platform.architecture()[0]}")
print(f"CPU Logical Cores: {os.cpu_count()}")

## 2. Virtual Environment Isolation Mechanics

Python determines if it is running inside an isolated virtual environment by comparing:
- `sys.prefix`: The prefix path of the current environment.
- `sys.base_prefix`: The prefix path of the base/system Python installation.

If `sys.prefix != sys.base_prefix`, you are inside a **virtual environment**!

In [ ]:
import os
import sys

is_virtualenv = sys.prefix != sys.base_prefix

print("=== Virtual Environment Check ===")
print(f"sys.prefix      : {sys.prefix}")
print(f"sys.base_prefix : {sys.base_prefix}")
print(f"Is Isolated?    : {is_virtualenv}")
print(f"VIRTUAL_ENV env : {os.environ.get('VIRTUAL_ENV', 'Not set (Using Global Python)')}")

## 3. How Python Finds Modules: `sys.path`

Whenever you run `import my_module`, Python iterates through the list of directories in `sys.path` in order until it finds a matching file or folder.

1. **Index 0:** The directory of the running script (or current working directory in interactive mode).
2. **Standard Library:** Built-in Python modules (e.g., `math`, `json`, `pathlib`).
3. **Site-Packages:** Where third-party packages installed via `uv` or `pip` reside.

In [ ]:
import sys

print("=== sys.path Search Chain ===")
for idx, path in enumerate(sys.path, start=1):
    print(f"[{idx:02d}] {path}")

### Experiment: Dynamic Path Resolution
Watch what happens when we look up where the standard library and third-party modules are loaded from using the `__file__` attribute.

In [ ]:
import json
import pathlib

print(f"json location    : {json.__file__}")
print(f"pathlib location : {pathlib.__file__}")

## 4. Modern Project Architecture: Why `src-layout` Wins

Compare the two directory structures:

```
Flat Layout (Risky):                 src Layout (Recommended):
my_project/                          my_project/
├── my_app/                          ├── src/
│   └── core.py                      │   └── my_app/
├── tests/                           │       └── core.py
└── pyproject.toml                   ├── tests/
                                     └── pyproject.toml
```

In the flat layout, running `pytest` automatically adds the root directory to `sys.path[0]`. This allows tests to import `my_app` directly from source, hiding missing package builds or faulty packaging configurations.

In the `src-layout`, your tests **must** install the package properly (`uv sync` or `uv pip install -e .`) to test the real installed package.

## 5. Python AST (Abstract Syntax Tree) & How Linters Like Ruff Work

How does `ruff` or `flake8` detect bugs without actually executing the code?
They parse your Python source code into an **Abstract Syntax Tree (AST)** and check for dangerous tree patterns (like mutable default arguments).

In [ ]:
import ast

flawed_code = """
def process_items(items=[]):
    items.append('new')
    return items
"""

# Parse Python code into AST
tree = ast.parse(flawed_code)

print("=== AST Structure (Dump) ===")
print(ast.dump(tree, indent=2))

# Search for mutable default arguments in AST
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        for default in node.args.defaults:
            if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                print(f"\n[LINTER ALERT] Function '{node.name}' contains mutable default argument of type {type(default).__name__} at line {node.lineno}!")

## 6. Next Steps

Now that you understand runtime mechanics and AST inspection:
1. Review `TROUBLESHOOTING_AND_EDGE_CASES.md` for real-world developer pitfalls.
2. Complete the challenges in `SELF_ASSESSMENT_AND_CHALLENGES.md`.
3. Build the mini project in `modern_project_template/` following `PROJECT_GUIDE.md`!